# Lab 14: Aggregation with GroupBy and Apply

## Objectives

This lab demonstrates how to perform data aggregation using Pandas.

Students will learn how to:

- Group patient records by hospital
- Calculate statistical measures like average stay duration
- Use `.apply()` to create calculated columns
- Chain multiple operations such as filter → groupby → aggregation
- Analyze healthcare datasets using pandas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create reproducible dataset
np.random.seed(42)

hospitals = ['City General', 'Metro Health', 'Regional Medical', 'Community Care', 'University Hospital']
departments = ['Emergency', 'Cardiology', 'Orthopedics', 'Pediatrics', 'Surgery']
insurance_types = ['Private', 'Medicare', 'Medicaid', 'Uninsured']

n_patients = 1000

patient_data = {
    'patient_id': range(1, n_patients + 1),
    'hospital': np.random.choice(hospitals, n_patients),
    'department': np.random.choice(departments, n_patients),
    'age': np.random.randint(18, 85, n_patients),
    'stay_duration': np.random.randint(1, 15, n_patients),
    'insurance_type': np.random.choice(insurance_types, n_patients),
    'admission_cost': np.random.randint(1000, 50000, n_patients)
}

df = pd.DataFrame(patient_data)

print("Dataset Overview")
print(df.head())

In [ ]:
print("Data Types")
print(df.dtypes)

print("\nBasic Statistics")
print(df.describe())

print("\nPatients per hospital")
print(df['hospital'].value_counts())

In [ ]:
# Average stay by hospital
avg_stay = df.groupby('hospital')['stay_duration'].mean()

print("Average Stay Duration by Hospital")
print(avg_stay)

avg_stay_df = avg_stay.reset_index()
avg_stay_df.columns = ['Hospital', 'Average_Stay']

print(avg_stay_df)

In [ ]:
hospital_stats = df.groupby('hospital')['stay_duration'].agg([
    'count',
    'mean',
    'median',
    'std',
    'min',
    'max'
]).round(2)

hospital_stats.columns = [
    'Patient_Count',
    'Avg_Stay',
    'Median_Stay',
    'Std_Stay',
    'Min_Stay',
    'Max_Stay'
]

print(hospital_stats)

In [ ]:
plt.figure(figsize=(10,5))

avg_stay.plot(kind='bar')

plt.title("Average Stay Duration by Hospital")
plt.xlabel("Hospital")
plt.ylabel("Days")

plt.show()

In [ ]:
def categorize_stay(days):
    if days <= 3:
        return "Short"
    elif days <= 7:
        return "Medium"
    else:
        return "Long"

df['stay_category'] = df['stay_duration'].apply(categorize_stay)

print(df[['stay_duration','stay_category']].head())

In [ ]:
def hospital_efficiency(group):
    return pd.Series({
        "avg_stay": group['stay_duration'].mean(),
        "avg_cost": group['admission_cost'].mean(),
        "patient_count": len(group)
    })

efficiency = df.groupby('hospital').apply(hospital_efficiency)

print(efficiency)

In [ ]:
result = (
    df
    .query("stay_duration > 5")
    .groupby("hospital")["admission_cost"]
    .mean()
    .round(2)
)

print("Average cost for long stay patients")
print(result)

In [ ]:
top_departments = (
    df
    .groupby(['hospital','department'])
    .agg({
        'admission_cost':'mean',
        'stay_duration':'mean',
        'patient_id':'count'
    })
    .rename(columns={'patient_id':'patient_count'})
    .sort_values('admission_cost',ascending=False)
    .head(10)
)

print(top_departments)

## Conclusion

In this lab we learned:

- How to use **GroupBy** for aggregation
- How to apply **custom functions with `.apply()`**
- How to build **data analysis pipelines using method chaining**

These techniques are widely used in **data science, healthcare analytics, and business intelligence**.